In [1]:
import torch

print(f"🖥️ CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")
!pip install transformers accelerate datasets optimum bitsandbytes huggingface
print("✅ Listo.")



import sys
import torch
import transformers
import accelerate
import datasets
import optimum
import bitsandbytes
from huggingface_hub import __version__ as hf_version

print("📦 Versiones del entorno — guardar esto si algo explota en el futuro:")
print(f"  Python:          {sys.version.split()[0]}")
print(f"  PyTorch:         {torch.__version__}")
print(f"  CUDA:            {torch.version.cuda}")
print(f"  transformers:    {transformers.__version__}")
print(f"  accelerate:      {accelerate.__version__}")
print(f"  datasets:        {datasets.__version__}")
print(f"  bitsandbytes:    {bitsandbytes.__version__}")
print(f"  huggingface_hub: {hf_version}")

print("\n# Para reproducir este entorno exacto:")
print(f"# !pip install transformers=={transformers.__version__} accelerate=={accelerate.__version__} datasets=={datasets.__version__}  bitsandbytes=={bitsandbytes.__version__}")

🖥️ CUDA: 12.6 | PyTorch: 2.8.0+cu126
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.4 MB/s eta 0:00:00:00:0100:01
✅ Listo.
📦 Versiones del entorno — guardar esto si algo explota en el futuro:
  Python:          3.12.12
  PyTorch:         2.8.0+cu126
  CUDA:            12.6
  transformers:    4.57.1
  accelerate:      1.11.0
  datasets:        4.4.2
  bitsandbytes:    0.49.2
  huggingface_hub: 0.36.0

# Para reproducir este entorno exacto:
# !pip install transformers==4.57.1 accelerate==1.11.0 datasets==4.4.2  bitsandbytes==0.49.2


In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"GPUs disponibles: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} - {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

MODEL_ID = "mistralai/Mistral-7B-v0.1"   # modelo original en HuggingFace
NEW_MODEL_NAME = "Mistral-7B-4bit-Ignacio"  # nombre con el que se guardará localmente y en el Hub

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

# Configuración de cuantización NF4 (Normal Float 4-bit)
# Es el formato de 4-bit con mejor relación calidad/tamaño disponible
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4 > INT4 en calidad para LLMs
    bnb_4bit_compute_dtype=torch.float16, # los cálculos internos siguen en FP16
    bnb_4bit_use_double_quant=True,      # cuantiza también los parámetros de cuantización, ahorra ~0.4 bits extra
)

# device_map="balanced" reparte las capas equitativamente entre las 2 T4
# max_memory deja 1.5GB libre en cada GPU para los activations durante el forward pass
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="balanced",
    max_memory={0: "12GiB", 1: "12GiB"},
)

# Verificamos que el reparto entre GPUs quedó bien (13 capas en cada T4 aprox.)
print("\n📊 Distribución del modelo:")
for name, device in model.hf_device_map.items():
    print(f"  {name}: GPU {device}")

# Guardamos localmente antes de subir al Hub
# Esto genera los archivos .safetensors + config.json + tokenizer
model.save_pretrained(NEW_MODEL_NAME)
tokenizer.save_pretrained(NEW_MODEL_NAME)
print(f"✅ Guardado en: {NEW_MODEL_NAME}")


# Subimos el modelo cuantizado al Hub para que sea accesible públicamente
print("🚀 Subiendo modelo a HuggingFace Hub...")
model.push_to_hub(NEW_MODEL_NAME)
tokenizer.push_to_hub(NEW_MODEL_NAME)
print(f"✅ Disponible en: https://huggingface.co/tu_usuario/{NEW_MODEL_NAME}")

GPUs disponibles: 2
  GPU 0: Tesla T4 - 15.6 GB
  GPU 1: Tesla T4 - 15.6 GB


tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Disponible en: https://huggingface.co/tu_usuario/Mistral-7B-4bit-Ignacio


In [4]:
import torch
from datasets import load_dataset
from tqdm import tqdm

torch.cuda.empty_cache()

# La perplejidad (PPL) mide qué tan bien el modelo predice texto
# Cuanto más baja, mejor. Mistral-7B original tiene ~5.25 en WikiText-2
# Recibe el modelo como parámetro para poder reutilizarla en la celda 6
def evaluar_perplejidad(mdl, batch_size=4):
    print("📚 Descargando WikiText-2 Test...")
    test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    
    # Unimos todos los textos del test set en una sola secuencia larga
    encodings = tokenizer("\n\n".join(test["text"]), return_tensors="pt")

    max_length = 2048  # ventana de contexto máxima por paso
    stride = 512       # cuánto avanzamos en cada paso (overlap para mejor estimación)
    seq_len = encodings.input_ids.size(1)

    # Pre-generamos todas las ventanas para poder iterar en batches
    windows = []
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc  # tokens nuevos en esta ventana (sin overlap)
        windows.append((begin_loc, end_loc, trg_len))
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    nlls = []  # negative log-likelihoods de cada ventana
    for i in tqdm(range(0, len(windows), batch_size)):
        batch = windows[i:i+batch_size]
        for begin_loc, end_loc, trg_len in batch:
            # Enviamos a cuda:0 porque ahí están las capas de entrada del modelo
            input_ids = encodings.input_ids[:, begin_loc:end_loc].to("cuda:0")
            target_ids = input_ids.clone()
            # Marcamos el overlap con -100 para que no se incluya en la loss
            target_ids[:, :-trg_len] = -100

            with torch.no_grad():  # sin gradientes, solo inferencia
                outputs = mdl(input_ids, labels=target_ids)
                nlls.append(outputs.loss.cpu())  # movemos a CPU para liberar VRAM

            del input_ids, target_ids, outputs

        # Limpiamos caché una vez por batch, no por ventana (más eficiente)
        torch.cuda.empty_cache()

    # PPL = exp(promedio de NLL) — fórmula estándar
    return torch.exp(torch.stack(nlls).mean()).item()

# Guardamos el resultado en variable global para reutilizarlo en celda 6
ppl_cuantizado = evaluar_perplejidad(model)
print(f"📊 Perplejidad modelo 4-bit: {ppl_cuantizado:.2f}")

📚 Descargando WikiText-2 Test...


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

100%|██████████| 163/163 [38:42<00:00, 14.25s/it]

📊 Perplejidad modelo 4-bit: 4.79


In [5]:
# ARC-Challenge es un benchmark de preguntas de ciencias de nivel escolar
# Mide razonamiento, no solo predicción de texto como la perplejidad
!pip install lm-eval

import lm_eval
from lm_eval.models.huggingface import HFLM

print("🧠 Ejecutando ARC-Challenge...")

# HFLM es un wrapper que adapta nuestro modelo al formato que espera lm_eval
hflm_model = HFLM(
    pretrained=model,
    tokenizer=tokenizer,
    batch_size=1  # batch_size bajo para no quedarnos sin VRAM
)

# limit=100 evalúa solo 100 ejemplos para que no tarde horas
# num_fewshot=0 = zero-shot, sin ejemplos de contexto
results = lm_eval.simple_evaluate(
    model=hflm_model,
    tasks=["arc_challenge"],
    num_fewshot=0,
    limit=100
)

print(results)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 83.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 7.1 MB/s eta 0:00:00
  Created wheel for sqlitedict: filename=sqlitedict-2.1.0-py3-none-any.whl size=16862 sha256=5de540bb7df8f5c6e4d096f898e4916d00d72657112de268b66febd5d720bdba
  Stored in directory: /root/.cache/pip/wheels/7a/6f/21/fc016aef45ffcabe27129a2252f061387cbf278d2086225a64
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5568 sha256=9c7c7cde29642e3f1cc15028f751ded7f3858431e80271c6c4cc92f0eb9ec7c3
  Stored in directory: /root/.cache/pip/wheels/5b/79/fb/d25928e599c7e11fe4e00d32048cd74933f34

🧠 Ejecutando ARC-Challenge...


README.md: 0.00B [00:00, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…):   0%|          | 0.00/190k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

Running loglikelihood requests: 100%|██████████| 400/400 [01:26<00:00,  4.64it/s]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


{'results': {'arc_challenge': {'alias': 'arc_challenge', 'acc,none': 0.49, 'acc_stderr,none': 0.05024183937956913, 'acc_norm,none': 0.46, 'acc_norm_stderr,none': 0.05009082659620332}}, 'group_subtasks': {'arc_challenge': []}, 'configs': {'arc_challenge': {'task': 'arc_challenge', 'tag': ['ai2_arc'], 'dataset_path': 'allenai/ai2_arc', 'dataset_name': 'ARC-Challenge', 'training_split': 'train', 'validation_split': 'validation', 'test_split': 'test', 'doc_to_text': 'Question: {{question}}\nAnswer:', 'doc_to_target': '{{choices.label.index(answerKey)}}', 'unsafe_code': False, 'doc_to_choice': '{{choices.text}}', 'description': '', 'target_delimiter': ' ', 'fewshot_delimiter': '\n\n', 'fewshot_config': {'sampler': 'default', 'split': None, 'process_docs': None, 'fewshot_indices': None, 'samples': None, 'doc_to_text': 'Question: {{question}}\nAnswer:', 'doc_to_choice': '{{choices.text}}', 'doc_to_target': '{{choices.label.index(answerKey)}}', 'gen_prefix': None, 'fewshot_delimiter': '\n\n', 

In [6]:
import torch, time

# ─────────────────────────────────────────────
# BENCHMARK 1 — Velocidad de inferencia (tokens/s) + VRAM
# ─────────────────────────────────────────────
PROMPTS = [
    "Explain the theory of relativity in simple terms:",
    "Write a Python function that sorts a list of dictionaries by a key:",
    "What are the main causes of the French Revolution?",
    "Describe the process of photosynthesis step by step:",
]

def benchmark_velocidad(mdl, prompts=PROMPTS, max_new_tokens=128, runs=3):
    print("\n⚡ BENCHMARK: Velocidad de inferencia")
    print("="*55)

    todos_los_tps = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
        n_input_tokens = inputs.input_ids.shape[1]

        # Calentamiento (el primer forward siempre es más lento por compilación JIT)
        with torch.no_grad():
            _ = mdl.generate(**inputs, max_new_tokens=16, do_sample=False)

        tps_runs = []
        for _ in range(runs):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad():
                out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - t0

            n_new = out.shape[1] - n_input_tokens
            tps_runs.append(n_new / elapsed)

        media = sum(tps_runs) / len(tps_runs)
        todos_los_tps.append(media)
        short_prompt = prompt[:45] + "..." if len(prompt) > 45 else prompt
        print(f"  Prompt: \"{short_prompt}\"")
        print(f"  → {media:.1f} tok/s  (promedio {runs} runs, {max_new_tokens} tokens nuevos)")

    promedio_global = sum(todos_los_tps) / len(todos_los_tps)
    print(f"\n  🏁 Throughput promedio global: {promedio_global:.1f} tok/s")
    return promedio_global

def benchmark_vram(mdl):
    print("\n💾 BENCHMARK: Uso de VRAM")
    print("="*55)
    total_vram_usada = 0
    for i in range(torch.cuda.device_count()):
        used  = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i} ({torch.cuda.get_device_name(i)}): {used:.2f} GB / {total:.1f} GB  ({used/total*100:.1f}%)")
        total_vram_usada += used
    print(f"  Total VRAM modelo: {total_vram_usada:.2f} GB")
    return total_vram_usada

tps_4bit  = benchmark_velocidad(model)
vram_4bit = benchmark_vram(model)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



⚡ BENCHMARK: Velocidad de inferencia


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  Prompt: "Explain the theory of relativity in simple te..."
  → 10.9 tok/s  (promedio 3 runs, 128 tokens nuevos)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  Prompt: "Write a Python function that sorts a list of ..."
  → 10.9 tok/s  (promedio 3 runs, 128 tokens nuevos)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  Prompt: "What are the main causes of the French Revolu..."
  → 10.9 tok/s  (promedio 3 runs, 128 tokens nuevos)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  Prompt: "Describe the process of photosynthesis step b..."
  → 10.9 tok/s  (promedio 3 runs, 128 tokens nuevos)

  🏁 Throughput promedio global: 10.9 tok/s

💾 BENCHMARK: Uso de VRAM
  GPU 0 (Tesla T4): 1.73 GB / 15.6 GB  (11.1%)
  GPU 1 (Tesla T4): 2.41 GB / 15.6 GB  (15.4%)
  Total VRAM modelo: 4.15 GB


In [7]:
# ─────────────────────────────────────────────
# BENCHMARK 2 — Tabla resumen comparativa
# Referencia FP16 tomada del paper oficial Mistral-7B (Jiang et al., 2023)
# ─────────────────────────────────────────────
import json

REFERENCIA_FP16 = {
    "PPL WikiText-2":            5.25,   # reportado por Mistral AI
    "ARC-Challenge (0-shot)":    0.595,  # accuracy reportada
    "Throughput (tok/s)":        None,   # depende del hardware
    "VRAM (GB)":                 14.0,   # ~14 GB en FP16
}

try:
    arc_acc = results["results"]["arc_challenge"]["acc,none"]
except Exception:
    arc_acc = None  # si la celda ARC aún no se ejecutó

resultados_4bit = {
    "PPL WikiText-2":            ppl_cuantizado if "ppl_cuantizado" in dir() else "pendiente",
    "ARC-Challenge (0-shot)":    round(arc_acc, 3) if arc_acc else "pendiente",
    "Throughput (tok/s)":        round(tps_4bit, 1) if "tps_4bit" in dir() else "pendiente",
    "VRAM (GB)":                 round(vram_4bit, 2) if "vram_4bit" in dir() else "pendiente",
}

print("\n📊 TABLA COMPARATIVA: Mistral-7B FP16  vs  4-bit NF4")
print("═"*62)
print(f"  {'Métrica':<30} {'FP16 (ref.)':<15} {'4-bit NF4'}")
print("─"*62)

for metrica in resultados_4bit:
    ref = REFERENCIA_FP16.get(metrica, "N/A")
    val = resultados_4bit[metrica]
    ref_str = str(ref) if ref is not None else "N/A"
    val_str = str(val)

    indicador = ""
    if isinstance(ref, float) and isinstance(val, float):
        if "PPL" in metrica:
            indicador = " ✅" if val <= ref * 1.05 else " ⚠️  +{:.1f}%".format((val/ref-1)*100)
        elif "Throughput" in metrica:
            indicador = " 🚀"
        elif "VRAM" in metrica:
            indicador = f" ✅ ahorra {ref - val:.1f} GB"
        else:
            indicador = " ✅" if val >= ref * 0.98 else " ⚠️  -{:.1f}%".format((1-val/ref)*100)

    print(f"  {metrica:<30} {ref_str:<15} {val_str}{indicador}")

print("─"*62)
print("\n  ✅ = sin degradación significativa  |  ⚠️  = degradación notable")

# Guardar JSON para la model card del Hub
benchmark_json = {
    "model": NEW_MODEL_NAME,
    "base_model": MODEL_ID,
    "quantization": "NF4 4-bit (bitsandbytes double quant)",
    "hardware": "2x Tesla T4 16GB",
    "results": resultados_4bit
}
with open(f"{NEW_MODEL_NAME}/benchmark_results.json", "w") as f:
    json.dump(benchmark_json, f, indent=2)
print(f"\n💾 Guardado en {NEW_MODEL_NAME}/benchmark_results.json")


📊 TABLA COMPARATIVA: Mistral-7B FP16  vs  4-bit NF4
══════════════════════════════════════════════════════════════
  Métrica                        FP16 (ref.)     4-bit NF4
──────────────────────────────────────────────────────────────
  PPL WikiText-2                 5.25            4.794521331787109 ✅
  ARC-Challenge (0-shot)         0.595           0.49 ⚠️  -17.6%
  Throughput (tok/s)             N/A             10.9
  VRAM (GB)                      14.0            4.15 ✅ ahorra 9.8 GB
──────────────────────────────────────────────────────────────

  ✅ = sin degradación significativa  |  ⚠️  = degradación notable

💾 Guardado en Mistral-7B-4bit-Ignacio/benchmark_results.json
